# BigQuery Context — Six Approaches to Table Discovery

A step-by-step walkthrough of the six discovery strategies this repo compares.
Each one takes the same question and produces the same `RerankerResponse`, so
you can watch exactly where they diverge: **what metadata each one puts in front
of the reranker**.

Runs against the live corpus created by `bq-context ensure-infra`. Nothing here
writes to the experiment — it is read-only apart from Gemini calls.

**Prerequisites**

```bash
make install
export GOOGLE_CLOUD_PROJECT=your-project
bq-context ensure-infra        # once, ~20 min
bq-context preflight --tier 3  # confirm enrichment reached the capsule
```

## Setup

In [1]:
import json
from pathlib import Path

from dotenv import load_dotenv

from bq_context.config import TIERS, ExperimentConfig
from bq_context.context_cache import TableCache
from bq_context.runtime import TierContext, get_datasets, get_scoped_tables, tier_scope

# Reads GOOGLE_CLOUD_PROJECT from the repo-root .env, or from the environment.
# Deliberately not defaulted to a project id: this notebook shipped with one
# hard-coded, so anyone else running it silently pointed at a project they
# could not read, and the first failure was a permission error naming nothing.
load_dotenv(Path.cwd().parent / ".env")

config = ExperimentConfig.from_env()
config.configure_adk_env()  # ADK builds its own genai client from the environment

print("project:", config.project)
print("agent model:", config.agent_model, "| tool model:", config.tool_model)
print("gemini endpoint:", config.locations.gemini, " <- global, not regional")

project: hybrid-vertex
agent model: gemini-3.6-flash | tool model: gemini-3.5-flash-lite
gemini endpoint: global  <- global, not regional


### Pin a tier

Every run is scoped to **exactly one** tier dataset. All four hold
identically-named tables and scoring matches on the short name, so a run that
saw two tiers at once would silently score against the wrong corpus.

`TierContext` carries the scope, the config, and the per-tier metadata cache.
It lives in a `ContextVar`, which is what lets 24 shards run in parallel.

In [2]:
TIER = 3  # 0 schema · 1 +profiling · 2 +glossary · 3 +table-level aspect

bootstrap = TierContext.build(config, TIER, TableCache.empty())
with tier_scope(bootstrap):
    datasets = get_datasets()
    scoped = {d: get_scoped_tables(d) for d in datasets}

cache = TableCache.build(config, datasets, scoped)
ctx = TierContext.build(config, TIER, cache)

print("scope:", datasets)
print("tables cached:", len(cache))

scope: ['bigquery_context_tier3']
tables cached: 15


### The question\n\nA `multi-table-disparate` case — the hard kind, where the answer needs tables that share no obvious name.

In [3]:
QUESTION = "Which US counties have the most weather stations per capita?"

# Ground truth for this question, from experiments/questions.json
with Path("../experiments/questions.json").open() as f:
    questions = {q["id"]: q for q in json.load(f)}
truth = questions["multi-disp-q3"]
print(QUESTION)
print("must_have:    ", truth["relevance"]["must_have"])
print("nice_to_have: ", truth["relevance"].get("nice_to_have", []))

Which US counties have the most weather stations per capita?
must_have:     ['us_counties', 'weather_stations', 'population_by_zip_2010']
nice_to_have:  ['zip_codes']


---
## The APIs — one table, three different views

Before the approaches, look at what each API actually returns for a single
table. This is the whole experiment in miniature: the approaches differ mainly
in **which of these they feed the reranker**.

In [4]:
EXAMPLE_TABLE = "weather_stations"
EXAMPLE_DATASET = config.tier_dataset(TIER)
EXAMPLE_FULL_ID = f"{config.project}.{EXAMPLE_DATASET}.{EXAMPLE_TABLE}"
print(EXAMPLE_FULL_ID)

hybrid-vertex.bigquery_context_tier3.weather_stations


### 1. BigQuery `get_table` — schema only\n\nWhat Approach 1 sees. Column names, types, modes. No profiling, no business meaning.

In [5]:
from google.cloud import bigquery

bq_client = bigquery.Client(project=config.project)
table = bq_client.get_table(EXAMPLE_FULL_ID)

print(f"{len(table.schema)} columns, description: {table.description or '(none)'}")
for f in table.schema[:6]:
    print(f"  {f.name:<24} {f.field_type:<10} {(f.description or '')[:46]}")

11 columns, description: Global Historical Climatology Network weather station inventory. Station locations with latitude, longitude, elevation, and name.
  id                       STRING     
  latitude                 FLOAT      
  longitude                FLOAT      
  elevation                FLOAT      
  state                    STRING     
  name                     STRING     


### 2. Knowledge Catalog `lookup_entry` — schema + catalog aspects\n\nWhat Approach 2 sees. Adds catalog aspects, but **no data profiling**.

In [6]:
from google.cloud import dataplex_v1
from google.protobuf import json_format

catalog = dataplex_v1.CatalogServiceClient()
entry_name = config.dataplex_entry_name(EXAMPLE_DATASET, EXAMPLE_TABLE)

entry = catalog.lookup_entry(
    request=dataplex_v1.LookupEntryRequest(
        name=f"projects/{config.project}/locations/{config.locations.catalog}",
        entry=entry_name,
        view=dataplex_v1.EntryView.FULL,
    )
)
# MessageToDict needs the raw protobuf; there is no public accessor for it.
aspects = sorted(json_format.MessageToDict(entry._pb).get("aspects", {}))  # noqa: SLF001
print("aspect keys:")
for a in aspects:
    print("  ", a)

aspect keys:
   655216118709.global.bigquery-view
   655216118709.global.data-profile
   655216118709.global.schema


### 3. Knowledge Catalog `lookup_context` — the LLM-ready capsule

What Approaches 3, 4 and 5 see. Schema **plus** per-column profiling
(`nullRatio`, `distinctValues`, `sampleValues`) **plus** glossary definitions.

Note where glossary data lives: a per-column **`terms`** key, carrying the
term's display name and description — *not* a top-level `related_terms` object.
Searching for the wrong key here is how an earlier analysis in this repo
concluded tier 2 was dead when it was not.

In [7]:
from bq_context.context_cache.util_lookup_context import lookup_context

capsule = json.loads(lookup_context(config, [entry_name]))[0]
print("capsule keys:", sorted(capsule))

col = (
    next(c for c in capsule["schema"] if c.get("terms"))
    if any(c.get("terms") for c in capsule["schema"])
    else capsule["schema"][0]
)
print(json.dumps(col, indent=2)[:700])

capsule keys: ['ancestors', 'bigqueryView', 'catalogEntry', 'createTime', 'description', 'name', 'resource', 'schema', 'simpleName', 'type', 'updateTime']
{
  "name": "id",
  "type": "STRING",
  "mode": "NULLABLE",
  "terms": "GHCN Station; Weather observation site in the Global Historical Climatology Network, identified by a unique station ID.",
  "dataProfile": {
    "nullRatio": 0.0,
    "distinctValues": 99.95,
    "sampleValues": [
      "ZA000067541",
      "WZ004822290",
      "WZ004451000",
      "WA012064960",
      "WA011037970",
      "WA010086600",
      "WA010026470",
      "WA009612470",
      "WA009195050",
      "WA009158370"
    ]
  }
}


#### The enrichment ladder

The same 15 tables exist in all four tier datasets. Only the catalog enrichment
differs — that replication **is** the ablation.

In [8]:
def enrichment_summary(tier: int) -> dict:
    boot = TierContext.build(config, tier, TableCache.empty())
    with tier_scope(boot):
        ds = get_datasets()
        sc = {d: get_scoped_tables(d) for d in ds}
    c = TableCache.build(config, ds, sc)
    profiled = terms = 0
    aspects = set()
    for e in c.entries.values():
        cap = json.loads(e.detailed)
        for column in cap.get("schema", []):
            profiled += "dataProfile" in column
            terms += bool(column.get("terms"))
        aspects |= {k for k in ("guidelines", "overview") if k in cap}
    return {
        "bytes": len(c.all_detailed()),
        "profiled_cols": profiled,
        "glossary_cols": terms,
        "aspects": sorted(aspects) or ["—"],
    }


print(f"{'tier':<6}{'bytes':>10}{'profiled':>10}{'glossary':>10}  aspects")
for t in TIERS:
    row = enrichment_summary(t)
    cols = f"{row['profiled_cols']:>10}{row['glossary_cols']:>10}"
    print(f"{t:<6}{row['bytes']:>10,}{cols}  {','.join(row['aspects'])}")

tier       bytes  profiled  glossary  aspects


0         49,089         0         0  —


1        118,275       209         0  —


2        119,882       209        18  —


3        122,346       209        18  overview


> **Why tier 3 shows `overview` and not `guidelines`.** Tier 3 is meant to add a
> `guidelines` aspect. Reading that aspect type requires
> `dataplex.aspectTypes.get` on Google's own `dataplex-types` project, which
> returns 403 **even for a near-Owner principal** — it is an availability
> restriction, not a gap in your IAM. `ensure-infra` therefore falls back to the
> `overview` aspect, which carries the same text and *does* reach the capsule.
> Do not spend an afternoon granting roles to fix this.


> **Caveat worth knowing.** The capsule truncates each schema to **25 columns**
> by default. On a wide table that silently drops glossary annotations past the
> cut — 6 of our 24 term links are lost this way. `all_schema_fields=true`
> recovers them at roughly 1.8x the capsule size.

In [9]:
wide_entry = config.dataplex_entry_name(config.tier_dataset(2), "hurricanes")
for label, opts in (
    ("default", {"format": "json"}),
    ("all_schema_fields=true", {"format": "json", "all_schema_fields": "true"}),
):
    req = dataplex_v1.LookupContextRequest(
        name=f"projects/{config.project}/locations/{config.locations.catalog}",
        resources=[wide_entry],
        options=opts,
    )
    res = json.loads(catalog.lookup_context(request=req).context)["resources"][0]
    cols = res.get("schema", [])
    print(
        f"  {label:<24} {len(cols):>4} columns, "
        f"{sum(1 for c in cols if c.get('terms'))} with glossary terms"
    )

  default                    25 columns, 0 with glossary terms


  all_schema_fields=true    153 columns, 5 with glossary terms


### brief vs detailed

The cache stores two views of the same capsule. The **brief** is *subtractive* —
it strips only the heavy per-column `dataProfile`, so cheap high-signal
enrichments (glossary terms, guidelines) survive into LLM pre-filtering.

In [10]:
brief_bytes, detail_bytes = len(cache.all_briefs()), len(cache.all_detailed())
n_brief = len(json.loads(cache.all_briefs()))
n_detail = len(json.loads(cache.all_detailed()))

print(f"brief    total {brief_bytes:>8,} bytes  (~{brief_bytes // n_brief:,}/table)")
print(f"detailed total {detail_bytes:>8,} bytes  (~{detail_bytes // n_detail:,}/table)")
print(f"\nratio: detailed is {detail_bytes / brief_bytes:.1f}x the brief")

brief    total   52,397 bytes  (~3,493/table)
detailed total  122,346 bytes  (~8,156/table)

ratio: detailed is 2.3x the brief


---
## The six approaches

Each one below follows the same three steps:

1. **Discover** — gather candidate tables and their metadata
2. **Preview** — see exactly what the reranker will receive
3. **Rerank** — call Gemini and inspect the ranking

The reranker is shared, so any difference in output traces to step 1.

In [11]:
from bq_context.reranker.util_rerank import call_reranker
from bq_context.schemas import RerankerResponse
from bq_context.usage import usage_scope


def rerank(metadata: str, method: str) -> RerankerResponse:
    """Call the shared reranker and report what it cost."""
    with usage_scope() as usage:
        result = call_reranker(
            config=config,
            question=QUESTION,
            candidate_metadata=metadata,
            discovery_method=method,
            top_k=config.top_k,
        )
    print(f"{method}: {usage.total_tokens:,} tokens, {len(result.ranked_tables)} tables")
    for t in result.ranked_tables:
        hit = "*" if t.table_id.rsplit(".", 1)[-1] in truth["relevance"]["must_have"] else " "
        print(f"  {hit} #{t.rank} {t.table_id.rsplit('.', 1)[-1]:<32} {t.confidence:.2f}")
    return result


results = {}

## Approach 1 — BQ Metadata Tools (`agent_bq_tools`)\n\nLLM tool loop over the BigQuery API. No catalog at all. The control for *'is a plain metadata loop enough?'*

### Step 1 — discover via BigQuery

In [12]:
bq_parts = []
for t in bq_client.list_tables(f"{config.project}.{EXAMPLE_DATASET}"):
    tbl = bq_client.get_table(f"{config.project}.{EXAMPLE_DATASET}.{t.table_id}")
    cols = ", ".join(f"{f.name} ({f.field_type})" for f in tbl.schema[:12])
    bq_parts.append(
        f"Table: {config.project}.{EXAMPLE_DATASET}.{t.table_id}\n"
        f"Description: {tbl.description or '(none)'}\nColumns: {cols}"
    )
print(f"{len(bq_parts)} tables described")

15 tables described


### Step 2 — what the reranker receives

In [13]:
print(next(p for p in bq_parts if EXAMPLE_TABLE in p)[:600])

Table: hybrid-vertex.bigquery_context_tier3.weather_stations
Description: Global Historical Climatology Network weather station inventory. Station locations with latitude, longitude, elevation, and name.
Columns: id (STRING), latitude (FLOAT), longitude (FLOAT), elevation (FLOAT), state (STRING), name (STRING), gsn_flag (STRING), hcn_crn_flag (STRING), wmoid (INTEGER), source_url (STRING), etl_timestamp (TIMESTAMP)


### Step 3 — rerank

In [14]:
results["bq_tools"] = rerank("\n\n".join(bq_parts), "bq_tools")

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


bq_tools: 4,760 tokens, 3 tables
  * #1 weather_stations                 0.95
  * #2 us_counties                      0.90
    #3 county_natality                  0.75


## Approach 2 — Knowledge Catalog Search (`agent_kc_search`)

Semantic search narrows the corpus, then `lookup_entry` fetches metadata for
each hit. The first approach with a real **retrieval** step.

### Step 1 — scoped semantic search

In [15]:
from bq_context.discovery_common import search_entries_scoped

with tier_scope(ctx):
    hits, stats = search_entries_scoped(QUESTION)

print("stats:", stats)
for h in hits:
    print("  ", h.table_id.rsplit(".", 1)[-1])

stats: {'raw_search_count': 6, 'out_of_scope_dropped': 0, 'page_size': 20}
   county_natality
   air_quality_annual_summary
   weather_stations
   us_counties
   austin_bikeshare_stations
   citibike_stations


> The query is built bare — no parentheses around the question. Wrapping it
> makes the parser **silently drop the `parent:` predicate**, leaking
> out-of-scope tables. `out_of_scope_dropped` should always be 0.

### Step 2 — fetch metadata per hit, and preview

In [16]:
search_parts = []
for h in hits:
    e = catalog.lookup_entry(
        request=dataplex_v1.LookupEntryRequest(
            name=f"projects/{config.project}/locations/{config.locations.catalog}",
            entry=h.entry_name,
            view=dataplex_v1.EntryView.FULL,
        )
    )
    d = json_format.MessageToDict(e._pb)  # noqa: SLF001  (see above)
    keep = {k: v for k, v in d.get("aspects", {}).items() if "schema" in k or "storage" in k}
    search_parts.append(
        f"Table: {h.table_id}\nDescription: {h.description}\n{json.dumps(keep)[:1200]}"
    )
print(search_parts[0][:600] if search_parts else "(no hits)")

Table: hybrid-vertex.bigquery_context_tier3.county_natality
Description: CDC WONDER natality (birth) statistics by US county and year. Each row aggregates births with average mother age, gestational age, birth weight, and pre-pregnancy BMI, keyed by county FIPS code.
{"655216118709.global.schema": {"aspectType": "projects/655216118709/locations/global/aspectTypes/schema", "createTime": "2026-09-22T01:35:22.735661Z", "updateTime": "2026-09-22T01:35:22.735661Z", "data": {"fields": [{"dataType": "DATE", "name": "Year", "mode": "NULLABLE", "metadataType": "DATETIME", "description": "Year"}, {"data


### Step 3 — rerank

In [17]:
results["kc_search"] = rerank("\n\n".join(search_parts), "kc_search")

kc_search: 5,346 tokens, 2 tables
  * #1 weather_stations                 0.95
  * #2 us_counties                      0.95


## Approach 3 — Knowledge Catalog Context (`agent_kc_context`)

No retrieval at all. Ship the **entire** cached corpus capsule to the reranker
and let it choose. Fastest to run, most expensive in tokens.

### Step 1 and 2 — the whole corpus, and what that costs

In [18]:
kc_context_metadata = cache.all_detailed()
print(f"{len(cache)} tables, {len(kc_context_metadata):,} bytes to the reranker")

15 tables, 122,346 bytes to the reranker


### Step 3 — rerank

In [19]:
results["kc_context"] = rerank(kc_context_metadata, "kc_context")

kc_context: 45,072 tokens, 5 tables
  * #1 weather_stations                 0.99
  * #2 us_counties                      0.95
    #3 county_natality                  0.85
  * #4 population_by_zip_2010           0.70
    #5 zip_codes                        0.65


## Approach 4 — Context Pre-Filter (`agent_context_prefilter`)

Hybrid: an LLM reads the cheap **briefs** and nominates candidates, then only
those get reranked on full detail. Two LLM calls instead of one.

### Step 1 — the LLM nominates from briefs

In [20]:
from google import genai
from google.genai import types

client = genai.Client(vertexai=True, project=config.project, location=config.locations.gemini)

prompt = (
    f"Question: {QUESTION}\n\n"
    f"Table briefs:\n{cache.all_briefs()}\n\n"
    "Return ONLY a JSON array of the fully-qualified table_id strings "
    "that could help answer the question."
)
resp = client.models.generate_content(
    model=config.tool_model,
    contents=prompt,
    config=types.GenerateContentConfig(response_mime_type="application/json", temperature=0.0),
)
nominated = json.loads(resp.text)
print(f"nominated {len(nominated)} of {len(cache)}:")
for n in nominated:
    print("  ", n.rsplit(".", 1)[-1])

nominated 4 of 15:
   weather_stations
   us_counties
   population_by_zip_2010
   zip_codes


### Step 2 — fetch detail for nominees only

In [21]:
prefilter_metadata = cache.detailed_for(nominated)
print(f"{len(prefilter_metadata):,} bytes (vs {len(cache.all_detailed()):,} for the whole corpus)")

25,858 bytes (vs 122,346 for the whole corpus)


### Step 3 — rerank

In [22]:
results["context_prefilter"] = rerank(prefilter_metadata, "context_prefilter")

context_prefilter: 12,445 tokens, 4 tables
  * #1 weather_stations                 0.95
  * #2 us_counties                      0.95
  * #3 population_by_zip_2010           0.85
    #4 zip_codes                        0.85


## Approach 5 — Semantic Context (`agent_semantic_context`)

Approach 2's search combined with Approach 3's cached capsules: narrow by
search, then enrich from cache instead of making N `lookup_entry` calls.

### Step 1 and 2 — search, then cache lookup

In [23]:
semantic_metadata = cache.detailed_for([h.table_id for h in hits])
print(f"{len(hits)} hits -> {len(semantic_metadata):,} bytes from cache (no extra API calls)")

6 hits -> 56,461 bytes from cache (no extra API calls)


### Step 3 — rerank

In [24]:
results["semantic_context"] = rerank(semantic_metadata, "semantic_context")

semantic_context: 22,288 tokens, 3 tables
  * #1 weather_stations                 0.99
  * #2 us_counties                      0.95
    #3 county_natality                  0.85


## Approach 6 — Search Direct (`agent_search_direct`)

The control. Semantic search's own relevance order **is** the ranking — no
reranker, no LLM, no tokens. Isolates exactly what reranking contributes.

In [25]:
from bq_context.schemas import RankedTable

n = len(hits)
results["search_direct"] = RerankerResponse(
    question=QUESTION,
    top_k=config.top_k,
    ranked_tables=[
        RankedTable(
            table_id=h.table_id,
            rank=i + 1,
            confidence=1.0 - (0.5 * i / n) if n else 0.0,
            reasoning="Semantic search relevance order.",
            discovery_method="search_direct",
        )
        for i, h in enumerate(hits)
    ],
    notes="No reranking applied.",
)

print("search_direct: 0 tokens, no LLM call")
for t in results["search_direct"].ranked_tables:
    hit = "*" if t.table_id.rsplit(".", 1)[-1] in truth["relevance"]["must_have"] else " "
    print(f"  {hit} #{t.rank} {t.table_id.rsplit('.', 1)[-1]:<32} {t.confidence:.2f}")

search_direct: 0 tokens, no LLM call
    #1 county_natality                  1.00
    #2 air_quality_annual_summary       0.92
  * #3 weather_stations                 0.83
  * #4 us_counties                      0.75
    #5 austin_bikeshare_stations        0.67
    #6 citibike_stations                0.58


---
## Cross-approach comparison

Rows are tables, columns are approaches. `*` marks a `must_have`. This is the
same view the orchestrator's compare agent builds from shared session state.

In [26]:
order = [
    "bq_tools",
    "kc_search",
    "kc_context",
    "context_prefilter",
    "semantic_context",
    "search_direct",
]
must = set(truth["relevance"]["must_have"])

ranks = {
    a: {t.table_id.rsplit(".", 1)[-1]: t.rank for t in results[a].ranked_tables}
    for a in order
    if a in results
}
tables = sorted(
    {t for r in ranks.values() for t in r},
    key=lambda t: (t not in must, min(r.get(t, 99) for r in ranks.values())),
)

print(f"{'':<34}" + "".join(f"{a[:9]:>11}" for a in order))
for t in tables:
    mark = "* " if t in must else "  "
    row = "".join(
        f"{('#' + str(ranks[a][t])) if t in ranks.get(a, {}) else '—':>11}" for a in order
    )
    print(f"{mark}{t:<32}{row}")

print(
    f"\n{'recall of must_have':<34}"
    + "".join(f"{len(must & set(ranks.get(a, {}))) / len(must):>11.0%}" for a in order)
)

                                     bq_tools  kc_search  kc_contex  context_p  semantic_  search_di
* weather_stations                         #1         #1         #1         #1         #1         #3
* us_counties                              #2         #2         #2         #2         #2         #4
* population_by_zip_2010                    —          —         #4         #3          —          —
  county_natality                          #3          —         #3          —         #3         #1
  air_quality_annual_summary                —          —          —          —          —         #2
  zip_codes                                 —          —         #5         #4          —          —
  austin_bikeshare_stations                 —          —          —          —          —         #5
  citibike_stations                         —          —          —          —          —         #6

recall of must_have                       67%        67%       100%       100%        67% 

---
## Running them as real ADK agents

Everything above unrolls each approach by hand so you can see the metadata. In
the experiment they run as ADK agents against an `InMemoryRunner`, one isolated
session per cell — which is what the shard runner does 3,000 times.

In [27]:
from bq_context.runner.cells import AdkCellExecutor
from bq_context.runner.models import ShardSpec

spec = ShardSpec(
    experiment_id="notebook",
    tier=TIER,
    approach="semantic_context",
    question_ids=["multi-disp-q3"],
    runs=1,
    code_version="notebook",
)

# Jupyter already runs an event loop, so `await` directly here.
# `asyncio.run()` raises "cannot be called from a running event loop".
with tier_scope(ctx):
    cell = await AdkCellExecutor(spec)(truth, 0)

print(f"status={cell.status}  latency={cell.latency_s}s  tokens={cell.reranker_total_tokens:,}")
print("ranked:", [t["table_id"].rsplit(".", 1)[-1] for t in cell.ranked_tables])

status=ok  latency=6.428s  tokens=22,072
ranked: ['weather_stations', 'us_counties', 'county_natality']


### From here

```bash
bq-context run-shard -e demo --tier 3 --approach semantic_context --limit 5 --runs 1
bq-context merge -e demo && bq-context score -e demo
```

Full command reference: [`experiments/README.md`](../experiments/README.md).

**Two caveats before you read tier numbers from your own run** — the Dataplex
semantic index takes time to converge after `ensure-infra`, and shards run in
tier order, so an early run can make enrichment look like it improves retrieval
when it is only measuring elapsed time. `bq-context preflight` warns on this.
See [`docs/notes/full-run-results.md`](../docs/notes/full-run-results.md).